# Tune what the agent remembers

Compare history windows, summaries, recall filters, result limits, similarity thresholds, and character budgets.
The output is the **actual context produced by the library's providers**, without calling an LLM.

Build the solution in Release, then start a fresh C# kernel in `notebooks/csharp`. No Redis service, credentials,
notebook 00, or OpenAI key is needed. [memory-lab.csx](memory-lab.csx) supplies a fixed synthetic session and six
memories across Avery and Riley. Its similarity scores are deliberately assigned teaching inputs, not measured
embedding similarities. This experiment explains controls; it does not benchmark real recall quality or model answers.

The provider inspection hooks are experimental in the pinned Agent Framework 1.19.0 API; these cells explicitly suppress its `MAAI001` diagnostic.


In [ ]:
#pragma warning disable MAAI001 // Inspect the experimental Agent Framework provider hooks.
#nullable enable
#r "nuget: Microsoft.Agents.AI, 1.19.0"
#r "nuget: Microsoft.Extensions.DependencyInjection, 10.0.12"
#r "nuget: ModelContextProtocol, 2.2.0"
#r "../../src/Redis.Iris.AgentMemory/bin/Release/net8.0/Redis.Iris.AgentMemory.dll"
#r "../../src/Redis.Iris.ContextRetriever.Modeling/bin/Release/net8.0/Redis.Iris.ContextRetriever.Modeling.dll"
#r "../../src/Redis.Iris.ContextRetriever/bin/Release/net8.0/Redis.Iris.ContextRetriever.dll"
#r "../../src/Redis.Iris.AgentFramework/bin/Release/net8.0/Redis.Iris.AgentFramework.dll"

using System;
using System.Collections.Generic;
using System.IO;
using System.Linq;
using System.Text.Json;
using System.Text.Json.Serialization;
using System.Threading;
using System.Threading.Tasks;
using Microsoft.Agents.AI;
using Redis.Iris.AgentFramework;
using Microsoft.Extensions.AI;
using Microsoft.Extensions.DependencyInjection;
using Redis.Iris.AgentMemory;
using Redis.Iris.ContextRetriever;
using Redis.Iris.ContextRetriever.Modeling;

using System.Net;
using System.Net.Http;
using System.Diagnostics;

In [ ]:
#pragma warning disable MAAI001 // Inspect the experimental Agent Framework provider hooks.
#load "memory-lab.csx"

## Inspect the fixed evidence

The fixture holds both users, but every scoped search below must select Avery. The two fixed questions have
different score orderings. An unfamiliar query gets score 0.10 in this fixture. In a real service, embeddings,
extraction, indexing, and changing data determine the rankings.

In [ ]:
#pragma warning disable MAAI001 // Inspect the experimental Agent Framework provider hooks.
foreach (var row in MemoryLabTransport.Records)
    Console.WriteLine($"{row.Memory.Id,-16} owner={row.Memory.OwnerId,-5} seat={row.SeatScore:F2} hotel={row.HotelScore:F2} type={row.Memory.MemoryType}: {row.Memory.Text}");

## History limits and summary budgets are independent

`MaxMessagesToRetrieve` limits recent events, not the summary or current user prompt. `MaxSummaryCharacters`
bounds only summary text; its untrusted-context label adds characters. Summaries come from the service: enabling
them cannot create a missing summary. Limits are applied locally after the session response arrives, so they do
not reduce the server response size.

Each scenario uses a fresh Agent Framework session to keep provider state independent. All fixture history is
synthetic; display real message contents only in an appropriately controlled environment.

In [ ]:
#pragma warning disable MAAI001 // Inspect the experimental Agent Framework provider hooks.
using (var http = new HttpClient(new MemoryLabTransport()))
{
    var client = MemoryLab.Client(http);
    foreach (var setting in new[]
    {
        (Name: "Two recent events", Recent: 2, Summary: false, SummaryBudget: 120),
        (Name: "Four recent events + summary", Recent: 4, Summary: true, SummaryBudget: 120),
        (Name: "Two events + clipped summary", Recent: 2, Summary: true, SummaryBudget: 40),
    })
    {
        var history = new RedisIrisChatHistoryProvider(client, _ => MemoryLab.State(), options: new()
        {
            MaxMessagesToRetrieve = setting.Recent, IncludeSessionSummary = setting.Summary,
            MaxSummaryCharacters = setting.SummaryBudget,
        });
        var messages = (await history.InvokingAsync(new(MemoryLab.Agent, await MemoryLab.Agent.CreateSessionAsync(),
            Array.Empty<ChatMessage>()))).ToList();
        MemoryLab.Check(messages.Count == setting.Recent + (setting.Summary ? 1 : 0), "Unexpected history count.");
        Console.WriteLine($"\n{setting.Name}: {messages.Count} messages, {messages.Sum(message => message.Text?.Length ?? 0)} text characters");
        MemoryLab.Show(messages);
    }
}

## Recall thresholds, filters, limits, and context budgets

Run the same two questions through six configurations. A higher threshold can remove relevant results; a lower
threshold admits weaker matches. A topic or memory-type filter can intentionally exclude an otherwise high-scoring
record. `Limit` controls the requested result count. `MaxContextCharacters` includes the recalled-context header
and list formatting, but excludes history and the user prompt; it may truncate the final memory mid-sentence.
Characters are not tokens. The provider does not automatically fetch another page when the context budget is small.

The request ledger and provenance IDs distinguish “no candidates” from “retrieved but clipped.” IDs are useful
here because the data is synthetic; they do not belong in production metric dimensions.

In [ ]:
#pragma warning disable MAAI001 // Inspect the experimental Agent Framework provider hooks.
var experiments = new[]
{
    (Name: "Broad", Threshold: .30, Limit: 4, Topic: (string?)null, Type: (string?)null, Budget: 600),
    (Name: "High threshold", Threshold: .90, Limit: 4, Topic: (string?)null, Type: (string?)null, Budget: 600),
    (Name: "No match", Threshold: .99, Limit: 4, Topic: (string?)null, Type: (string?)null, Budget: 600),
    (Name: "Flight preferences", Threshold: .30, Limit: 4, Topic: (string?)"flights", Type: (string?)"preference", Budget: 600),
    (Name: "Top one", Threshold: .30, Limit: 1, Topic: (string?)null, Type: (string?)null, Budget: 600),
    (Name: "Small context", Threshold: .30, Limit: 4, Topic: (string?)null, Type: (string?)null, Budget: 130),
};
var observations = new List<(string Question, string Setting, string Context, string[] IncludedIds)>();
foreach (var question in new[] { MemoryLab.SeatingQuestion, MemoryLab.HotelQuestion })
{
    foreach (var setting in experiments)
    {
        var transport = new MemoryLabTransport();
        using var http = new HttpClient(transport);
        var recall = new RedisIrisMemoryProvider(new RedisIrisLongTermMemorySearch(MemoryLab.Client(http)), _ => MemoryLab.State(), new()
        {
            FailClosed = true, MaxContextCharacters = setting.Budget,
            Search = new() { SimilarityThreshold = setting.Threshold, Limit = setting.Limit,
                Topics = setting.Topic is null ? null : new() { Eq = setting.Topic },
                MemoryType = setting.Type is null ? null : new() { Eq = setting.Type } },
        });
        var messages = (await recall.InvokingAsync(new(MemoryLab.Agent, await MemoryLab.Agent.CreateSessionAsync(),
            new[] { MemoryLab.Question(question) }))).ToList();
        var context = messages.SingleOrDefault(message => message.AdditionalProperties?.ContainsKey("redis.iris.provider") == true);
        var ids = context is null ? Array.Empty<string>() : (string[])context.AdditionalProperties!["redis.iris.memory_ids"]!;
        MemoryLab.Check(transport.LastSearch!.Text == question && transport.LastSearch.Filter!.OwnerId!.Eq == "avery", "Query or owner scope changed.");
        MemoryLab.Check(ids.All(id => id.StartsWith("avery-", StringComparison.Ordinal)), "Foreign memory included.");
        MemoryLab.Check((context?.Text?.Length ?? 0) <= setting.Budget, "Recall exceeded its character budget.");
        if (setting.Name == "No match") MemoryLab.Check(context is null, "High-threshold fixture should be empty.");
        if (setting.Name == "Top one") MemoryLab.Check(ids.Length == 1, "Expected one provenance ID.");
        if (setting.Name == "Flight preferences") MemoryLab.Check(ids.All(id => id is "avery-seat" or "avery-meal"), "Structured filter did not narrow results.");
        observations.Add((question, setting.Name, context?.Text ?? "", ids));
        Console.WriteLine($"{question,-30} {setting.Name,-20} included={ids.Length} chars={context?.Text?.Length ?? 0}/{setting.Budget} ids={string.Join(",", ids)}");
    }
}

## Inspect exactly what changed

Compare broad recall and a small budget for the seating question. The current prompt would also be passed to the
model, along with any history provider output and instructions configured by the host. These budgets do not cap
the entire model context. Provenance indicates a memory contributed text, not that the complete record fit.

In [ ]:
#pragma warning disable MAAI001 // Inspect the experimental Agent Framework provider hooks.
foreach (var observation in observations.Where(item => item.Question == MemoryLab.SeatingQuestion && item.Setting is "Broad" or "Small context"))
{
    Console.WriteLine($"\n{observation.Setting} — exact recalled message:");
    Console.WriteLine(observation.Context.Length == 0 ? "[no recalled message]" : observation.Context);
}
MemoryLab.Check(observations.Single(item => item.Question == MemoryLab.SeatingQuestion && item.Setting == "Broad").Context.Length >
    observations.Single(item => item.Question == MemoryLab.SeatingQuestion && item.Setting == "Small context").Context.Length,
    "Expected a visible budget reduction.");
Console.WriteLine("All tuning assertions passed; every HTTP fixture was disposed within its cell.");

## Transfer the experiment to your deployment

Use the owner-scoped searches from notebooks 01/05 with a small fixed question set and known expected memories.
Record whether expected memories reach the provider, inspect clipped text, and compare answers separately.
Do not reuse these hand-assigned scores as production thresholds. A valid request with empty results can mean
promotion is pending, filters exclude the record, or the threshold is too high; a failed request is a different
condition (notebook 08). No data was written or deleted by this notebook.